In [ ]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

[Introduction](introyt1_tutorial.html) \|\|
[Tensors](tensors_deeper_tutorial.html) \|\|
[Autograd](autogradyt_tutorial.html) \|\| **Building Models** \|\|
[TensorBoard Support](tensorboardyt_tutorial.html) \|\| [Training
Models](trainingyt.html) \|\| [Model Understanding](captumyt.html)

Building Models with PyTorch
============================

Follow along with the video below or on
[youtube](https://www.youtube.com/watch?v=OSqIP-mOWOI).



In [ ]:
# Run this cell to load the video
from IPython.display import display, HTML
html_code = """
<div style="margin-top:10px; margin-bottom:10px;">
  <iframe width="560" height="315" src="https://www.youtube.com/embed/OSqIP-mOWOI" frameborder="0" allow="accelerometer; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>
</div>
"""
display(HTML(html_code))



`torch.nn.Module` and `torch.nn.Parameter`  
`torch.nn.Module` 和 `torch.nn.Parameter`
------------------------------------------

In this video, we'll be discussing some of the tools PyTorch makes
available for building deep learning networks.  
在本视频中，我们将讨论 PyTorch 为构建深度学习网络提供的一些工具。

Except for `Parameter`, the classes we discuss in this video are all
subclasses of `torch.nn.Module`. This is the PyTorch base class meant to
encapsulate behaviors specific to PyTorch Models and their components.  
除了 `Parameter` 之外，我们在本视频中讨论的类都是 `torch.nn.Module` 的子类。这是 PyTorch 的基类，旨在封装特定于 PyTorch 模型及其组件的行为。

One important behavior of `torch.nn.Module` is registering parameters.
If a particular `Module` subclass has learning weights, these weights
are expressed as instances of `torch.nn.Parameter`. The `Parameter`
class is a subclass of `torch.Tensor`, with the special behavior that
when they are assigned as attributes of a `Module`, they are added to
the list of that modules parameters. These parameters may be accessed
through the `parameters()` method on the `Module` class.  
`torch.nn.Module` 的一个重要行为是注册参数。如果某个特定的 `Module` 子类具有可学习的权重，这些权重会被表示为 `torch.nn.Parameter` 的实例。`Parameter` 类是 `torch.Tensor` 的子类，具有一个特殊行为：当它们被赋值为 `Module` 的属性时，会被添加到该模块的参数列表中。这些参数可以通过 `Module` 类上的 `parameters()` 方法进行访问。

As a simple example, here's a very simple model with two linear layers
and an activation function. We'll create an instance of it and ask it to
report on its parameters:  
作为一个简单的例子，这里有一个非常简单的模型，包含两个线性层和一个激活函数。我们将创建一个实例，并让它报告其参数：


In [ ]:
import torch

class TinyModel(torch.nn.Module):
    
    def __init__(self):
        super().__init__()
        
        self.linear1 = torch.nn.Linear(100, 200)
        self.activation = torch.nn.ReLU()
        self.linear2 = torch.nn.Linear(200, 10)
        self.softmax = torch.nn.Softmax(dim=1)
    
    def forward(self, x):
        x = self.linear1(x)
        x = self.activation(x)
        x = self.linear2(x)
        x = self.softmax(x)
        return x

tinymodel = TinyModel()

print('The model:')
print(tinymodel)

print('\n\nJust one layer:')
print(tinymodel.linear2)

print('\n\nModel params:')
for param in tinymodel.parameters():
    print(param)

print('\n\nLayer params:')
for param in tinymodel.linear2.parameters():
    print(param)

This shows the fundamental structure of a PyTorch model: there is an
`__init__()` method that defines the layers and other components of a
model, and a `forward()` method where the computation gets done. Note
that we can print the model, or any of its submodules, to learn about
its structure.  
这展示了 PyTorch 模型的基本结构：有一个 `__init__()` 方法用于定义模型的层和其他组件，以及一个 `forward()` 方法用于执行具体的计算。请注意，我们可以打印模型或其任何子模块来了解其结构。

Common Layer Types  
常见的层类型
==================

Linear Layers  
线性层
-------------

The most basic type of neural network layer is a *linear* or *fully
connected* layer. This is a layer where every input influences every
output of the layer to a degree specified by the layer's weights. If a
model has *m* inputs and *n* outputs, the weights will be an *m* x *n*
matrix. For example:  
最基础的神经网络层是*线性层*（或称*全连接层*）。在这一层中，每个输入都会以层权重所指定的程度影响该层的每个输出。如果一个模型有 *m* 个输入和 *n* 个输出，那么权重将是一个 *m* x *n* 的矩阵。例如：

In [ ]:
lin = torch.nn.Linear(3, 2)
x = torch.rand(1, 3)
print('Input:')
print(x)

print('\n\nWeight and Bias parameters:')
for param in lin.parameters():
    print(param)

y = lin(x)
print('\n\nOutput:')
print(y)

If you do the matrix multiplication of `x` by the linear layer's
weights, and add the biases, you'll find that you get the output vector
`y`.  
如果你将 `x` 与线性层的权重进行矩阵乘法，并加上偏置（biases），你会发现得到的正是输出向量 `y`。

One other important feature to note: When we checked the weights of our
layer with `lin.weight`, it reported itself as a `Parameter` (which is a
subclass of `Tensor`), and let us know that it's tracking gradients with
autograd. This is a default behavior for `Parameter` that differs from
`Tensor`.  
另一个需要注意的重要特性：当我们使用 `lin.weight` 检查该层的权重时，它会显示为一个 `Parameter`（`Tensor` 的子类），并提示它正在通过 autograd 追踪梯度。这是 `Parameter` 区别于普通 `Tensor` 的默认行为。

Linear layers are used widely in deep learning models. One of the most
common places you'll see them is in classifier models, which will
usually have one or more linear layers at the end, where the last layer
will have *n* outputs, where *n* is the number of classes the classifier
addresses.  
线性层在深度学习模型中被广泛使用。最常见的应用场景之一是在分类器模型中，这类模型通常在末尾包含一个或多个线性层，其中最后一层的输出数量 *n* 对应分类器所处理的类别数。


Convolutional Layers  
卷积层
====================

*Convolutional* layers are built to handle data with a high degree of
spatial correlation. They are very commonly used in computer vision,
where they detect close groupings of features which the compose into
higher-level features. They pop up in other contexts too - for example,
in NLP applications, where a word's immediate context (that is, the
other words nearby in the sequence) can affect the meaning of a
sentence.  
*卷积层*专为处理具有高度空间相关性的数据而构建。它们在计算机视觉领域应用极为广泛，能够检测紧密组合的特征，并将其构建成更高级别的特征。卷积层在其他场景中也有应用——例如在自然语言处理（NLP）中，一个词的直接上下文（即序列中相邻的其他词）会影响整个句子的含义。

We saw convolutional layers in action in LeNet5 in an earlier video:  
我们在之前关于 LeNet5 的视频中已经看到了卷积层的实际应用：

In [ ]:
import torch.nn.functional as F


class LeNet(torch.nn.Module):

    def __init__(self):
        super().__init__()
        # 1 input image channel (black & white), 6 output channels, 5x5 square convolution
        # kernel
        self.conv1 = torch.nn.Conv2d(1, 6, 5)
        self.conv2 = torch.nn.Conv2d(6, 16, 3)
        # an affine operation: y = Wx + b
        self.fc1 = torch.nn.Linear(16 * 6 * 6, 120)  # 6*6 from image dimension
        self.fc2 = torch.nn.Linear(120, 84)
        self.fc3 = torch.nn.Linear(84, 10)

    def forward(self, x):
        # Max pooling over a (2, 2) window
        x = F.max_pool2d(F.relu(self.conv1(x)), (2, 2))
        # If the size is a square you can only specify a single number
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = x.view(-1, self.num_flat_features(x))
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

    def num_flat_features(self, x):
        size = x.size()[1:]  # all dimensions except the batch dimension
        num_features = 1
        for s in size:
            num_features *= s
        return num_features

Let's break down what's happening in the convolutional layers of this
model. Starting with `conv1`:  
让我们分解一下模型中卷积层的工作原理。首先从 `conv1` 开始：

-   LeNet5 is meant to take in a 1x32x32 black & white image. **The
    first argument to a convolutional layer's constructor is the number
    of input channels.** Here, it is 1. If we were building this model
    to look at 3-color channels, it would be 3.  
    LeNet5 的设计初衷是接收一张 1x32x32 的黑白图像。**卷积层构造函数的第一个参数是输入通道的数量。** 这里，输入通道数为 1。如果我们要构建一个用于查看 3 色通道（如 RGB）的模型，该值则为 3。
-   A convolutional layer is like a window that scans over the image,
    looking for a pattern it recognizes. These patterns are called
    *features,* and one of the parameters of a convolutional layer is
    the number of features we would like it to learn. **This is the
    second argument to the constructor is the number of output
    features.** Here, we're asking our layer to learn 6 features.  
    卷积层就像一个在图像上扫描、寻找它所识别的特定模式的窗口。这些模式被称为*特征*，而卷积层的一个参数就是我们希望它学习的特征数量。**这是构造函数的第二个参数，即输出特征的数量。** 在这里，我们要求我们的层学习 6 个特征。
-   Just above, I likened the convolutional layer to a window - but how
    big is the window? **The third argument is the window or kernel
    size.** Here, the "5" means we've chosen a 5x5 kernel. (If you want
    a kernel with height different from width, you can specify a tuple
    for this argument - e.g., `(3, 5)` to get a 3x5 convolution kernel.)  
    前面我将卷积层比作一个窗口，但这个窗口到底有多大呢？**第三个参数是窗口或卷积核的大小。** 这里的 "5" 表示我们选择了 5x5 的卷积核。（如果你想要一个高和宽不同的卷积核，可以为该参数指定一个元组，例如 `(3, 5)` 来获得一个 3x5 的卷积核。）

The output of a convolutional layer is an *activation map* - a spatial
representation of the presence of features in the input tensor. `conv1`
will give us an output tensor of 6x28x28; 6 is the number of features,
and 28 is the height and width of our map. (The 28 comes from the fact
that when scanning a 5-pixel window over a 32-pixel row, there are only
28 valid positions.)  
卷积层的输出是一个*激活图*，它是输入张量中特征存在的空间表示。`conv1` 将给我们一个 6x28x28 的输出张量；6 是特征的数量，28 是我们激活图的高度和宽度。（这个 28 的由来是，当一个 5 像素的窗口在 32 像素的行上扫描时，只有 28 个有效位置。）

We then pass the output of the convolution through a ReLU activation
function (more on activation functions later), then through a max
pooling layer. The max pooling layer takes features near each other in
the activation map and groups them together. It does this by reducing
the tensor, merging every 2x2 group of cells in the output into a single
cell, and assigning that cell the maximum value of the 4 cells that went
into it. This gives us a lower-resolution version of the activation map,
with dimensions 6x14x14.  
然后，我们将卷积的输出通过一个 ReLU 激活函数（稍后详细介绍激活函数），再通过一个最大池化层。最大池化层将激活图中彼此靠近的特征组合在一起。它通过降低张量的维度来实现这一点，将输出中每一个 2x2 的单元格组合并为一个单元格，并将该单元格的值指定为进入它的 4 个单元格中的最大值。这为我们提供了激活图的一个低分辨率版本，维度为 6x14x14。

Our next convolutional layer, `conv2`, expects 6 input channels
(corresponding to the 6 features sought by the first layer), has 16
output channels, and a 3x3 kernel. It puts out a 16x12x12 activation
map, which is again reduced by a max pooling layer to 16x6x6. Prior to
passing this output to the linear layers, it is reshaped to a 16 \* 6 \*
6 = 576-element vector for consumption by the next layer.  
我们的下一个卷积层 `conv2` 期望接收 6 个输入通道（对应于第一层寻找的 6 个特征），有 16 个输出通道，以及一个 3x3 的卷积核。它输出一个 16x12x12 的激活图，该激活图再次通过一个最大池化层缩减为 16x6x6。在将此输出传递给线性层之前，它被重塑为一个 16 \* 6 \* 6 = 576 个元素的向量，供下一层处理。

There are convolutional layers for addressing 1D, 2D, and 3D tensors.
There are also many more optional arguments for a conv layer
constructor, including stride length(e.g., only scanning every second or
every third position) in the input, padding (so you can scan out to the
edges of the input), and more. See the
[documentation](https://pytorch.org/docs/stable/nn.html#convolution-layers)
for more information.  
PyTorch 提供了用于处理 1D、2D 和 3D 张量的卷积层。卷积层构造函数还有许多其他可选参数，包括步幅长度（例如，只扫描输入中的第二个或第三个位置）、填充（因此你可以扫描到输入的边缘）等。更多信息请参阅<官方文档>。

Recurrent Layers  
循环层
================

*Recurrent neural networks* (or *RNNs)* are used for sequential data
-anything from time-series measurements from a scientific instrument to
natural language sentences to DNA nucleotides. An RNN does this by
maintaining a *hidden state* that acts as a sort of memory for what it
has seen in the sequence so far.  
*循环神经网络*（或 *RNNs*）用于处理序列数据，包括从科学仪器的时间序列测量值到自然语言句子再到 DNA 核苷酸序列。RNN 通过维持一个*隐藏状态*来实现这一点，该隐藏状态充当了它到目前为止在序列中看到的内容的一种记忆。

The internal structure of an RNN layer - or its variants, the LSTM (long
short-term memory) and GRU (gated recurrent unit) - is moderately
complex and beyond the scope of this video, but we'll show you what one
looks like in action with an LSTM-based part-of-speech tagger (a type of
classifier that tells you if a word is a noun, verb, etc.):  
RNN 层的内部结构，或者其变体 LSTM（长短期记忆）和 GRU（门控循环单元），是中等复杂的，超出了本视频的范围。但我们将向你展示一个基于 LSTM 的词性标注器（一种告诉你一个单词是名词、动词等的分类器）的实际应用：


In [ ]:
class LSTMTagger(torch.nn.Module):

    def __init__(self, embedding_dim, hidden_dim, vocab_size, tagset_size):
        super().__init__()
        self.hidden_dim = hidden_dim

        self.word_embeddings = torch.nn.Embedding(vocab_size, embedding_dim)

        # The LSTM takes word embeddings as inputs, and outputs hidden states
        # with dimensionality hidden_dim.
        self.lstm = torch.nn.LSTM(embedding_dim, hidden_dim)

        # The linear layer that maps from hidden state space to tag space
        self.hidden2tag = torch.nn.Linear(hidden_dim, tagset_size)

    def forward(self, sentence):
        embeds = self.word_embeddings(sentence)
        lstm_out, _ = self.lstm(embeds.view(len(sentence), 1, -1))
        tag_space = self.hidden2tag(lstm_out.view(len(sentence), -1))
        tag_scores = F.log_softmax(tag_space, dim=1)
        return tag_scores

The constructor has four arguments:

-   `vocab_size` is the number of words in the input vocabulary. Each
    word is a one-hot vector (or unit vector) in a
    `vocab_size`-dimensional space.
-   `tagset_size` is the number of tags in the output set.
-   `embedding_dim` is the size of the *embedding* space for the
    vocabulary. An embedding maps a vocabulary onto a low-dimensional
    space, where words with similar meanings are close together in the
    space.
-   `hidden_dim` is the size of the LSTM's memory.

The input will be a sentence with the words represented as indices of
one-hot vectors. The embedding layer will then map these down to an
`embedding_dim`-dimensional space. The LSTM takes this sequence of
embeddings and iterates over it, fielding an output vector of length
`hidden_dim`. The final linear layer acts as a classifier; applying
`log_softmax()` to the output of the final layer converts the output
into a normalized set of estimated probabilities that a given word maps
to a given tag.

If you'd like to see this network in action, check out the [Sequence
Models and LSTM
Networks](https://pytorch.org/tutorials/beginner/nlp/sequence_models_tutorial.html)
tutorial on pytorch.org.

Transformers
============

*Transformers* are multi-purpose networks that have taken over the state
of the art in NLP with models like BERT. A discussion of transformer
architecture is beyond the scope of this video, but PyTorch has a
`Transformer` class that allows you to define the overall parameters of
a transformer model - the number of attention heads, the number of
encoder & decoder layers, dropout and activation functions, etc. (You
can even build the BERT model from this single class, with the right
parameters!) The `torch.nn.Transformer` class also has classes to
encapsulate the individual components (`TransformerEncoder`,
`TransformerDecoder`) and subcomponents (`TransformerEncoderLayer`,
`TransformerDecoderLayer`). For details, check out the
[documentation](https://pytorch.org/docs/stable/nn.html#transformer-layers)
on transformer classes.

Other Layers and Functions
--------------------------

Data Manipulation Layers
========================

There are other layer types that perform important functions in models,
but don't participate in the learning process themselves.

**Max pooling** (and its twin, min pooling) reduce a tensor by combining
cells, and assigning the maximum value of the input cells to the output
cell (we saw this). For example:


构造函数包含四个参数：

-   `vocab_size` 是输入词汇表中的单词数量。每个单词都是一个 one-hot 向量（或单位向量），位于 `vocab_size` 维的空间中。
-   `tagset_size` 是输出标签集中的标签数量。
-   `embedding_dim` 是词汇表的*嵌入*空间大小。嵌入将词汇表映射到一个低维空间，其中语义相似的单词在空间中距离较近。
-   `hidden_dim` 是 LSTM 记忆单元的大小。

输入将是一个句子，单词表示为 one-hot 向量的索引。嵌入层随后会将其映射到 `embedding_dim` 维空间中。LSTM 接收这一序列的嵌入向量并进行迭代处理，输出一个长度为 `hidden_dim` 的向量。最后的线性层充当分类器；对最后一层的输出应用 `log_softmax()` 函数，可将其转换为一组归一化的估计概率，表示给定单词映射到给定标签的可能性。

如果你想看这个网络的实际运行效果，请查看 pytorch.org 上的 [序列模型和 LSTM 网络](https://pytorch.org/tutorials/beginner/nlp/sequence_models_tutorial.html) 教程。

Transformers (变换器)
============

*变换器* 是一种多功能网络，凭借 BERT 等模型在自然语言处理（NLP）领域的各项任务中取得了最先进的效果（SOTA）。关于变换器架构的详细讨论超出了本视频的范围，但 PyTorch 提供了一个 `Transformer` 类，允许你定义变换器模型的整体参数——例如注意力头的数量、编码器和解码器的层数、Dropout 和激活函数等。（你甚至可以使用这个单一的类，配合正确的参数，来构建 BERT 模型！）`torch.nn.Transformer` 类还提供了封装各个组件（`TransformerEncoder`、`TransformerDecoder`）和子组件（`TransformerEncoderLayer`、`TransformerDecoderLayer`）的类。详情请查看关于变换器类的 [文档](https://pytorch.org/docs/stable/nn.html#transformer-layers)。

其他层和函数
--------------------------

数据操作层
========================

还有一些其他类型的层在模型中执行重要功能，但其自身并不参与学习过程。

**最大池化**（及其孪生兄弟，最小池化）通过合并单元格来减小张量的尺寸，并将输入单元格中的最大值赋给输出单元格（这一点我们在前面已经看到）。例如：

In [ ]:
my_tensor = torch.rand(1, 6, 6)
print(my_tensor)

maxpool_layer = torch.nn.MaxPool2d(3)
print(maxpool_layer(my_tensor))

If you look closely at the values above, you'll see that each of the
values in the maxpooled output is the maximum value of each quadrant of
the 6x6 input.

**Normalization layers** re-center and normalize the output of one layer
before feeding it to another. Centering and scaling the intermediate
tensors has a number of beneficial effects, such as letting you use
higher learning rates without exploding/vanishing gradients.


In [ ]:
my_tensor = torch.rand(1, 4, 4) * 20 + 5
print(my_tensor)

print(my_tensor.mean())

norm_layer = torch.nn.BatchNorm1d(4)
normed_tensor = norm_layer(my_tensor)
print(normed_tensor)

print(normed_tensor.mean())

Running the cell above, we've added a large scaling factor and offset to
an input tensor; you should see the input tensor's `mean()` somewhere in
the neighborhood of 15. After running it through the normalization
layer, you can see that the values are smaller, and grouped around zero
- in fact, the mean should be very small (\> 1e-8).

This is beneficial because many activation functions (discussed below)
have their strongest gradients near 0, but sometimes suffer from
vanishing or exploding gradients for inputs that drive them far away
from zero. Keeping the data centered around the area of steepest
gradient will tend to mean faster, better learning and higher feasible
learning rates.

**Dropout layers** are a tool for encouraging *sparse representations*
in your model - that is, pushing it to do inference with less data.

Dropout layers work by randomly setting parts of the input tensor
*during training* - dropout layers are always turned off for inference.
This forces the model to learn against this masked or reduced dataset.
For example:


In [ ]:
my_tensor = torch.rand(1, 4, 4)

dropout = torch.nn.Dropout(p=0.4)
print(dropout(my_tensor))
print(dropout(my_tensor))

Above, you can see the effect of dropout on a sample tensor. You can use
the optional `p` argument to set the probability of an individual weight
dropping out; if you don't it defaults to 0.5.

Activation Functions
====================

Activation functions make deep learning possible. A neural network is
really a program - with many parameters - that *simulates a mathematical
function*. If all we did was multiple tensors by layer weights
repeatedly, we could only simulate *linear functions;* further, there
would be no point to having many layers, as the whole network would
reduce could be reduced to a single matrix multiplication. Inserting
*non-linear* activation functions between layers is what allows a deep
learning model to simulate any function, rather than just linear ones.

`torch.nn.Module` has objects encapsulating all of the major activation
functions including ReLU and its many variants, Tanh, Hardtanh, sigmoid,
and more. It also includes other functions, such as Softmax, that are
most useful at the output stage of a model.

Loss Functions
==============

Loss functions tell us how far a model's prediction is from the correct
answer. PyTorch contains a variety of loss functions, including common
MSE (mean squared error = L2 norm), Cross Entropy Loss and Negative
Likelihood Loss (useful for classifiers), and others.


如果你仔细观察上面的数值，你会发现最大池化（maxpooled）输出中的每个值，都是 6x6 输入张量对应象限内的最大值。

**归一化层（Normalization layers）**会在将某一层的输出传递给下一层之前，对其进行重新居中和归一化处理。对中间张量进行居中和缩放有许多益处，例如允许你使用更高的学习率，而不会出现梯度爆炸或梯度消失的问题。

运行上面的代码单元后，我们向输入张量添加了一个较大的缩放因子和偏移量；你应该能看到输入张量的 `mean()`（均值）大约在 15 左右。将其通过归一化层处理后，你可以看到数值变小了，并且聚集在零附近——事实上，均值应该非常小（> 1e-8）。

这非常有益，因为许多激活函数（下文会讨论）在接近 0 时具有最强的梯度，但当输入值使其远离零时，有时会出现梯度消失或梯度爆炸的问题。将数据保持在梯度最陡峭的区域附近，往往意味着更快、更好的学习效果以及更高的可用学习率。

**Dropout 层**是鼓励模型产生*稀疏表示*的一种工具——也就是说，促使模型在数据较少的情况下进行推理。

Dropout 层通过在*训练期间*随机将输入张量的部分元素置零来工作——Dropout 层在推理时始终处于关闭状态。这迫使模型针对这种被掩码或减少的数据集进行学习。例如：

在上面，你可以看到 Dropout 对样本张量的影响。你可以使用可选的 `p` 参数来设置单个权重被丢弃的概率；如果不设置，它默认为 0.5。

激活函数（Activation Functions）
====================

激活函数使深度学习成为可能。神经网络本质上是一个包含许多参数的程序，它*模拟一个数学函数*。如果我们仅仅是对张量反复乘以层权重，我们只能模拟*线性函数*；此外，拥有多层也就失去了意义，因为整个网络最终可以简化为单次矩阵乘法。在层与层之间插入*非线性*激活函数，使得深度学习模型能够模拟任意函数，而不仅仅是线性函数。

`torch.nn.Module` 包含了封装所有主要激活函数的对象，包括 ReLU 及其众多变体、Tanh、Hardtanh、Sigmoid 等。它还包含其他函数，例如 Softmax，这类函数在模型的输出阶段最为常用。

损失函数（Loss Functions）
==============

损失函数用于衡量模型的预测值与正确答案之间的差距。PyTorch 包含多种损失函数，包括常用的 MSE（均方误差 = L2 范数）、交叉熵损失（Cross Entropy Loss）和负对数似然损失（Negative Likelihood Loss，对分类器很有用）等。